# Day 099 Solution — Build Your Portfolio

In [ ]:
import pathlib, tempfile, os
from collections import Counter
from dataclasses import dataclass, field

@dataclass
class ProjectEntry:
    name: str; tagline: str; description: str; tech_stack: list
    github_url: str = ""; demo_url: str = ""; category: str = "AI Engineering"
    highlights: list = field(default_factory=list)

@dataclass
class PortfolioConfig:
    owner_name: str; title: str; bio: str; email: str; github_username: str
    linkedin_url: str = ""; projects: list = field(default_factory=list)

_P1 = ProjectEntry(
    name        = "AI Trading Bot",
    tagline     = "Paper-trading bot with sentiment + technical signals.",
    description = "Built over Days 89-96, this bot fetches OHLCV data, computes "
                  "technical indicators, scores news headlines with an LLM, applies "
                  "stop-loss and drawdown controls, and logs results daily.",
    tech_stack  = ["Python", "pandas", "Ollama", "SQLite"],
    github_url  = "https://github.com/testuser/ai-trading-bot",
    category    = "Finance",
    highlights  = ["Fully automated daily paper-trading loop",
                   "Kelly Criterion position sizing", "Stop-loss + drawdown gating"],
)
_P2 = ProjectEntry(
    name        = "Ops Agent",
    tagline     = "Autonomous multi-step ops agent with guardrails.",
    description = "Agent loop with tool routing, human-in-the-loop approval gates, "
                  "and task queue persistence.",
    tech_stack  = ["Python", "Ollama", "ChromaDB"],
    category    = "AI Agents",
    highlights  = ["Handles 5 operations autonomously", "Approval gate for destructive ops"],
)
_P3 = ProjectEntry(
    name        = "RAG Chatbot",
    tagline     = "Q&A chatbot grounded in your documents.",
    description = "Retrieval-augmented generation over a personal knowledge base.",
    tech_stack  = ["Python", "ChromaDB", "Ollama", "FastAPI"],
    github_url  = "https://github.com/testuser/rag-chatbot",
    demo_url    = "https://rag-chatbot.example.com",
    category    = "Text AI",
)

_CFG = PortfolioConfig(
    owner_name      = "Jane Doe",
    title           = "AI Engineer",
    bio             = "I build practical AI applications with Python. "
                      "100 days of AI engineering, shipped.",
    email           = "jane@example.com",
    github_username = "janedoe",
    linkedin_url    = "https://linkedin.com/in/janedoe",
    projects        = [_P1, _P2, _P3],
)
def _render_project_card(project):
    tech_tags = " ".join(f'<span class="tag">{t}</span>' for t in project.tech_stack)
    links = []
    if project.github_url: links.append(f'<a href="{project.github_url}">GitHub</a>')
    if project.demo_url:   links.append(f'<a href="{project.demo_url}">Demo</a>')
    links_html = " · ".join(links)
    card = (
        '      <div class="card">\n'
        f'        <h3>{project.name}</h3>\n'
        f'        <p class="category">{project.category}</p>\n'
        f'        <p>{project.tagline}</p>\n'
        f'        <div class="tags">{tech_tags}</div>\n'
    )
    if links_html:
        card += f'        <p class="links">{links_html}</p>\n'
    card += '      </div>'
    return card
def generate_portfolio_page(config):
    project_cards = "\n".join(_render_project_card(p) for p in config.projects)
    contact_parts = []
    if config.email:
        contact_parts.append(f'<a href="mailto:{config.email}">{config.email}</a>')
    if config.github_username:
        contact_parts.append(f'<a href="https://github.com/{config.github_username}">GitHub</a>')
    if config.linkedin_url:
        contact_parts.append(f'<a href="{config.linkedin_url}">LinkedIn</a>')
    contact_html = " · ".join(contact_parts)
    return (
        "<!DOCTYPE html>\n"
        '<html lang="en">\n'
        "<head>\n"
        '  <meta charset="UTF-8">\n'
        f"  <title>{config.owner_name} — {config.title}</title>\n"
        "  <style>\n"
        "    body { font-family: system-ui, sans-serif; margin: 0; }\n"
        "    header { background: #0f172a; color: white; padding: 60px 40px; }\n"
        "    h1 { font-size: 2.5rem; margin: 0 0 8px; }\n"
        "    .grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(320px, 1fr)); gap: 24px; }\n"
        "    .card { border: 1px solid #e2e8f0; border-radius: 12px; padding: 24px; }\n"
        "    .tag { background: #f1f5f9; padding: 2px 8px; border-radius: 4px; font-size: 0.8rem; }\n"
        "  </style>\n"
        "</head>\n"
        "<body>\n"
        "  <header>\n"
        f"    <h1>{config.owner_name}</h1>\n"
        f'    <p class="subtitle">{config.title}</p>\n'
        f'    <p class="bio">{config.bio}</p>\n'
        f'    <p class="contact">{contact_html}</p>\n'
        "  </header>\n"
        "  <main>\n"
        "    <h2>Projects</h2>\n"
        '    <div class="grid">\n'
        f"{project_cards}\n"
        "    </div>\n"
        "  </main>\n"
        "</body>\n"
        "</html>"
    )
def generate_case_study(project):
    tech_list  = "\n".join(f"- {t}" for t in project.tech_stack)
    highlights = (
        "\n".join(f"- {h}" for h in project.highlights)
        if project.highlights else "- See project README for details"
    )
    links = []
    if project.github_url: links.append(f"- GitHub: {project.github_url}")
    if project.demo_url:   links.append(f"- Demo: {project.demo_url}")
    links_text = "\n".join(links) if links else "- See GitHub profile"
    return (
        f"# {project.name}\n\n"
        f"**Category:** {project.category}\n\n"
        f"## Overview\n\n"
        f"{project.tagline}\n\n"
        f"{project.description}\n\n"
        f"## Tech Stack\n\n"
        f"{tech_list}\n\n"
        f"## Key Achievements\n\n"
        f"{highlights}\n\n"
        f"## Links\n\n"
        f"{links_text}\n"
    )
def generate_github_readme(config):
    project_lines = "\n".join(
        f"- **[{p.name}]({p.github_url or '#'})** — {p.tagline}"
        for p in config.projects
    )
    all_tech = []
    for p in config.projects: all_tech.extend(p.tech_stack)
    unique_tech = list(dict.fromkeys(all_tech))[:8]
    tech_line = " · ".join(unique_tech)
    linkedin_line = f"- LinkedIn: {config.linkedin_url}\n" if config.linkedin_url else ""
    return (
        f"# Hi, I'm {config.owner_name}\n\n"
        f"{config.bio}\n\n"
        f"## What I Build\n\n"
        f"I'm an **{config.title}** focused on building practical AI applications.\n\n"
        f"## Projects\n\n"
        f"{project_lines}\n\n"
        f"## Tech Stack\n\n"
        f"{tech_line}\n\n"
        f"## Contact\n\n"
        f"- Email: [{config.email}](mailto:{config.email})\n"
        f"{linkedin_line}"
    )
def summarize_portfolio(config):
    categories = [p.category for p in config.projects]
    tech = []
    for p in config.projects: tech.extend(p.tech_stack)
    top_tech = [item for item, _ in Counter(tech).most_common(5)]
    return {
        "n_projects":         len(config.projects),
        "categories":         sorted(set(categories)),
        "n_categories":       len(set(categories)),
        "top_tech":           top_tech,
        "total_tech_entries": len(tech),
    }
def export_portfolio(config, output_dir):
    out = pathlib.Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    written = []
    index_path = out / "index.html"
    index_path.write_text(generate_portfolio_page(config), encoding="utf-8")
    written.append(str(index_path))
    cases_dir = out / "case_studies"
    cases_dir.mkdir(exist_ok=True)
    for p in config.projects:
        slug = p.name.lower().replace(" ", "_").replace("-", "_")
        md_path = cases_dir / f"{slug}.md"
        md_path.write_text(generate_case_study(p), encoding="utf-8")
        written.append(str(md_path))
    return written


In [ ]:
import tempfile, pathlib

with tempfile.TemporaryDirectory() as tmp:
    written = export_portfolio(_CFG, tmp)
    out = pathlib.Path(tmp)

    # Assertions
    assert len(written) == 1 + len(_CFG.projects)
    assert "index.html" in written[0]
    index_content = (out / "index.html").read_text(encoding="utf-8")
    assert index_content.startswith("<!DOCTYPE html")
    assert _CFG.owner_name in index_content
    for p in _CFG.projects:
        assert f"<h3>{p.name}</h3>" in index_content

    for p in _CFG.projects:
        slug = p.name.lower().replace(" ", "_").replace("-", "_")
        md = (out / "case_studies" / f"{slug}.md").read_text(encoding="utf-8")
        assert md.startswith(f"# {p.name}")
        for t in p.tech_stack:
            assert f"- {t}" in md

    readme = generate_github_readme(_CFG)
    assert readme.startswith(f"# Hi, I'm {_CFG.owner_name}")
    for p in _CFG.projects:
        assert p.name in readme

    stats = summarize_portfolio(_CFG)
    assert stats["n_projects"] == 3
    assert stats["top_tech"][0] == "Python"
    assert "Finance" in stats["categories"]

    print("HTML:", len(index_content), "chars")
    print("Stats:", stats)
    print("\nSolution smoke-test passed.")
